# Handwritten Formula to LaTeX OCR - Training and evaluating

This notebook is a CPU-friendly copy of the training workflow for Google Colab.

Important:
- It is intended for pipeline validation and small experiments.
- Full fine-tuning on Colab CPU is impractical.
- This notebook uses reduced sample sizes and CPU-safe settings.

## 1. Setup

In [ ]:
# Check runtime
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("Expected runtime: CPU")

In [ ]:
# Install dependencies
%pip install -q torch torchvision
%pip install -q transformers>=4.45.0 accelerate>=0.25.0
%pip install -q peft>=0.7.0 datasets>=2.14.0
%pip install -q editdistance nltk sacrebleu wandb python-dotenv huggingface_hub

import nltk
nltk.download('punkt', quiet=True)

In [ ]:
# Clone repository
!git clone https://github.com/dmitryz1024/Multimodal_Reasoning_for_STEM.git
%cd Multimodal_Reasoning_for_STEM

In [ ]:
# Hugging Face login
from huggingface_hub import login

HF_TOKEN = ""  # Paste your token here if needed
if HF_TOKEN:
    login(HF_TOKEN)
    print("Logged in to Hugging Face")
else:
    print("HF_TOKEN not provided. Public datasets/models may still work.")

## 2. Quick Dataset Check

In [ ]:
from datasets import load_dataset

ds_latex_ocr = load_dataset("linxy/LaTeX_OCR", "human_handwrite")
print(ds_latex_ocr)
print(ds_latex_ocr["train"][0].keys())

## 3. CPU-Safe Training Setup

In [ ]:
from src.train import TrainConfig, train

config = TrainConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    primary_subset="human_handwrite",
    use_secondary=False,
    num_epochs=1,
    batch_size=1,
    eval_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    use_lora=True,
    lora_r=8,
    lora_alpha=16,
    load_in_4bit=False,
    load_in_8bit=False,
    bf16=False,
    fp16=False,
    gradient_checkpointing=False,
    max_length=256,
    max_samples_train=128,
    max_samples_val=32,
    logging_steps=1,
)

print("CPU-safe config:")
for key, value in config.__dict__.items():
    print(f"  {key}: {value}")
print("Training logs will appear below: model loading, dataset loading, step progress, epoch end, and final checkpoint save.")

In [ ]:
# Monitoring info before CPU training
from datetime import datetime
from pathlib import Path

run_name_cpu = "colab_cpu_smoke_test"
checkpoint_cpu = Path("checkpoints") / run_name_cpu / "final"
print("Monitoring")
print("- Current time:", datetime.now().isoformat(timespec="seconds"))
print("- Run name:", run_name_cpu)
print("- Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("- Expected checkpoint:", checkpoint_cpu.resolve())

In [ ]:
# Train a small CPU run
print("Starting CPU smoke-test training...")
model_latex_only, processor = train(
    config=config,
    run_name="colab_cpu_smoke_test",
    wandb_project=None,
)
print("CPU smoke-test finished. Checkpoint saved to ./checkpoints/colab_cpu_smoke_test/final")

## 4. Evaluation on the 70-example Test Slice

In [ ]:
from src.evaluate import run_full_evaluation, EvalConfig
import json

eval_config = EvalConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    subset="human_handwrite",
    num_samples=70,
)

results = run_full_evaluation(
    base_model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    checkpoint_latex_ocr="./checkpoints/colab_cpu_smoke_test/final",
    checkpoint_combined=None,
    config=eval_config,
)

with open("evaluation_results_cpu.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved:", "evaluation_results_cpu.json")
print(results)

## 5. Output Paths

In [ ]:
from pathlib import Path

print("Checkpoint:", Path("checkpoints/colab_cpu_smoke_test/final").resolve())
print("Evaluation:", Path("evaluation_results_cpu.json").resolve())